In [11]:
import pandas as pd
from pathlib import Path
import re
from tqdm import tqdm
import gemmi
import numpy as np
from cctbx import crystal

from mlindex.optimization.CandidateValidation import validate_candidate_known_bl
from mlindex.utilities.UnitCellTools import get_partial_unit_cell

In [12]:
base_dir = '/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/data/opxrd'
df = pd.read_json(Path(base_dir, 'CNRS_output_data_verified_final3.json'))

In [13]:
results_dir = '/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/comparisons/results'
ito_dir = Path(results_dir, 'ito')
dicvol_dir = Path(results_dir, 'dicvol')
treor_dir = Path(results_dir, 'treor')
gsas_dir = Path(results_dir, 'gsas')

In [14]:
def parse_ito(file_name):
    if not file_name.exists():
        return None

    in_reciprocal = False
    in_direct = False
    lines_indexed = []
    figure_of_merit = []
    unit_cells = []
    volumes = []
    centering = []

    with open(file_name, 'r') as f:
        for line in f:
            stripped = line.strip()

            # Detect section headers
            if 'Q(A)' in line and 'ZEROSHIFT' in line:
                in_reciprocal = True
                continue
            if 'ALFA' in line and 'VOLUME' in line:
                in_reciprocal = False
                in_direct = True
                continue

            # Parse data lines (non-empty after stripping)
            tokens = stripped.split()
            if not tokens:
                in_reciprocal = False
                in_direct = False
                continue

            if in_reciprocal:
                lines_indexed.append(int(float(tokens[6])))
                figure_of_merit.append(float(tokens[7]))
                centering.append(tokens[9])

            elif in_direct:
                if len(unit_cells) < len(lines_indexed):
                    values = [float(t) for t in tokens]
                    unit_cells.append(values[:6])
                    volumes.append(values[6])

    return pd.DataFrame({
        'Lines Indexed':   lines_indexed,
        'Figure of Merit': figure_of_merit,
        'Unit Cell':       unit_cells,
        'Volume':          volumes,
        'Centering':       centering,
    })

In [15]:
def parse_dicvol(file_name):
    if not file_name.exists():
        return None

    results = []
    current_params = None
    current_system = None
    parse_next_line = False

    with open(file_name, 'r') as f:
        for line_num, line in enumerate(f):
            if line_num >= 100_000:
                break
            # Detect crystal system
            if 'O R T H O R H O M B I C' in line:
                current_system = 'Orthorhombic'
            elif 'M O N O C L I N I C' in line:
                current_system = 'Monoclinic'
            elif 'T R I C L I N I C' in line:
                current_system = 'Triclinic'
            elif 'TETRAGONAL SYSTEM' in line:
                current_system = 'Tetragonal'
            elif 'HEXAGONAL SYSTEM' in line:
                current_system = 'Hexagonal'
            elif 'CUBIC SYSTEM' in line:
                current_system = 'Cubic'

            # If flagged, this line contains the triclinic parameters
            if parse_next_line:
                parse_next_line = False
                current_params = {'System': current_system}
                _extract_params(line, current_params)

            # Detect DIRECT PARAMETERS line
            elif 'DIRECT PARAMETERS' in line:
                if 'AND THEIR STANDARD DEVIATIONS' in line:
                    # Triclinic: parameters are on the next line
                    parse_next_line = True
                else:
                    # All other systems: parameters are on this line
                    current_params = {'System': current_system}
                    _extract_params(line, current_params)
                    if current_system == 'Orthorhombic':
                        current_params.setdefault('Alpha', 90.0)
                        current_params.setdefault('Beta', 90.0)
                        current_params.setdefault('Gamma', 90.0)

            # Capture first figure of merit M(n) and store the result
            if current_params and '1.- M(' in line:
                match = re.search(r'M\(\s*\d+\)\s*=\s*([\d.]+)', line)
                if match:
                    current_params['M'] = float(match.group(1))
                    results.append(current_params)
                    current_params = None

    if not results:
        return None

    return pd.DataFrame(results, columns=['System', 'A', 'B', 'C', 'Alpha', 'Beta', 'Gamma', 'Volume', 'M'])


def _extract_params(line, params):
    """Extract key=value pairs from a DIRECT PARAMETERS line into params dict."""
    for key, col_name in [('A=', 'A'), ('B=', 'B'), ('C=', 'C'),
                           ('ALP=', 'Alpha'), ('BET=', 'Beta'), ('BETA=', 'Beta'),
                           ('GAM=', 'Gamma'), ('VOL=', 'Volume'), ('VOLUME=', 'Volume')]:
        match = re.search(re.escape(key) + r'\s*([\d.]+)', line)
        if match:
            params[col_name] = float(match.group(1))

In [16]:
_TREOR_PARAM_RE = re.compile(r'([A-Z]+)\s*=\s*([\d.]+)')
_TREOR_MERIT_RE = re.compile(r'M\(\s*\d+\)\s*=\s*(\d+)')

def _is_corrupted_param_line(line):
    """Return True if a parameter line contains merged numbers (multiple decimals or asterisks)."""
    if '*' in line:
        return True
    # Split on whitespace and check each token for multiple decimal points
    # A merged value like '4.595938 63.338139' or '7.227516818.997131' will
    # appear as a token with more than one '.'  or as two tokens where the
    # surrounding context has too many values
    # More robust: check each '=' field's value portion for multiple dots
    for match in re.finditer(r'=\s*(\S+)', line):
        token = match.group(1)
        if token.count('.') > 1:
            return True
    return False

def parse_treor(file_name):
    if not file_name.exists():
        return None

    results = []
    current_params = None
    param_lines_remaining = 0
    merit_seen = False  # debounce the duplicate M() line

    with open(file_name, 'r') as f:
        for line_num, line in enumerate(f):
            if line_num >= 100_000:
                break
            stripped = line.lstrip()
            if not stripped:
                continue

            if '** CUBIC TEST *********************' in line:
                current_system = 'cubic'
            elif '** TETRAGONAL TEST ****************' in line:
                current_system = 'tetragonal'
            elif '** HEXAGONAL TEST *****************' in line:
                current_system = 'hexagonal'
            elif '** ORTHORHOMBIC TEST **************' in line:
                current_system = 'orthorhombic'
            elif '** MONOCLINIC TEST ****************' in line:
                current_system = 'monoclinic'
            elif '** TRICLINIC TEST *****************' in line:
                current_system = 'triclinic'

            # Skip HKL table lines (start with digit, minus, or dot)
            if stripped[0] in '-.' or (stripped[0].isdigit() and 'NUMBER' not in line and 'VERSION' not in line):
                continue

            # Detect start of a new solution block
            if 'NUMBER OF SINGLE INDEXED LINES' in line:
                current_params = {}
                param_lines_remaining = 0
                merit_seen = False
                continue

            # The three parameter lines follow "TOTAL NUMBER OF LINES"
            if 'TOTAL NUMBER OF LINES' in line:
                param_lines_remaining = 3
                continue

            if param_lines_remaining > 0:
                if current_params is not None:
                    if _is_corrupted_param_line(line):
                        current_params = None
                        param_lines_remaining = 0
                        continue
                    for match in _TREOR_PARAM_RE.finditer(line):
                        key = match.group(1)
                        val = float(match.group(2))
                        key_map = {'A': 'A', 'B': 'B', 'C': 'C',
                                   'ALFA': 'Alpha', 'BETA': 'Beta', 'GAMMA': 'Gamma'}
                        if key in key_map:
                            current_params[key_map[key]] = val
                param_lines_remaining -= 1
                continue

            if 'UNIT CELL VOLUME' in line and current_params is not None:
                match = re.search(r'([\d.]+)\s*A', line)
                if match:
                    current_params['Volume'] = float(match.group(1))
                continue

            # Capture M() — printed twice per solution, only store the first
            if current_params is not None and 'M(' in line and 'AV.EPS' in line:
                if not merit_seen:
                    match = _TREOR_MERIT_RE.search(line)
                    if match:
                        current_params['M'] = int(match.group(1))
                        merit_seen = True
                        # Only store if not already in results (avoid cycle duplicate)
                        already_stored = any(
                            r['A'] == current_params.get('A') and
                            r['B'] == current_params.get('B') and
                            r['C'] == current_params.get('C') and
                            r['Volume'] == current_params.get('Volume')
                            for r in results
                        )
                        if not already_stored:
                            current_params['lattice_system'] = current_system
                            results.append(dict(current_params))
                continue

    if not results:
        return None

    return pd.DataFrame(results, columns=['lattice_system', 'A', 'B', 'C', 'Alpha', 'Beta', 'Gamma', 'Volume', 'M'])

In [17]:
def parse_gsas(file_name):
    results = []
    if not file_name.exists():
        return None
    with open(file_name, 'r') as f:
        for line in f:
            stripped = line.lstrip()
            if not stripped:
                continue
            entry_results = {}
            M20, rest = line.split('(')
            entry_results['M20'] = float(M20.replace('\t', ''))
            unit_cell, sg = rest.split(')')
            entry_results['spacegroup'] = sg.strip()
            entry_results['a'] = float(unit_cell.split(',')[0])
            entry_results['b'] = float(unit_cell.split(',')[1])
            entry_results['c'] = float(unit_cell.split(',')[2])
            entry_results['alpha'] = float(unit_cell.split(',')[3])
            entry_results['beta'] = float(unit_cell.split(',')[4])
            entry_results['gamma'] = float(unit_cell.split(',')[5])
            sg = gemmi.SpaceGroup(entry_results['spacegroup'])
            entry_results['lattice_system'] = sg.crystal_system().name.lower()
            if entry_results['lattice_system'] == 'trigonal':
                entry_results['lattice_system'] = 'rhombohedral'
            centering = sg.centring_type()
            entry_results['bravais_lattice'] = None
            if entry_results['lattice_system'] == 'cubic':
                if centering == 'P':
                    entry_results['bravais_lattice'] = 'cP'
                elif centering == 'I':
                    entry_results['bravais_lattice'] = 'cI'
                elif centering == 'F':
                    entry_results['bravais_lattice'] = 'cF'
            elif entry_results['lattice_system'] == 'tetragonal':
                if centering == 'P':
                    entry_results['bravais_lattice'] = 'tP'
                elif centering == 'I':
                    entry_results['bravais_lattice'] = 'tI'
            elif entry_results['lattice_system'] == 'hexagonal':
                if centering == 'P':
                    entry_results['bravais_lattice'] = 'hP'
                elif centering == 'R':
                    entry_results['bravais_lattice'] = 'hR'
            elif entry_results['lattice_system'] == 'rhombohedral':
                entry_results['bravais_lattice'] = 'hR'
            elif entry_results['lattice_system'] == 'orthorhombic':
                if centering == 'P':
                    entry_results['bravais_lattice'] = 'oP'
                elif centering == 'I':
                    entry_results['bravais_lattice'] = 'oI'
                elif centering == 'F':
                    entry_results['bravais_lattice'] = 'oF'
                elif centering == 'C':
                    entry_results['bravais_lattice'] = 'oC'
            elif entry_results['lattice_system'] == 'monoclinic':
                if centering == 'P':
                    entry_results['bravais_lattice'] = 'mP'
                elif centering == 'I':
                    entry_results['bravais_lattice'] = 'mI'
                elif centering == 'C':
                    entry_results['bravais_lattice'] = 'mC'
            elif entry_results['lattice_system'] == 'triclinic':
                entry_results['bravais_lattice'] = 'aP'
            if entry_results['bravais_lattice'] is None:
                print(entry_results['lattice_system'], entry_results['spacegroup'])
            results.append(entry_results)
    return pd.DataFrame(results)


In [18]:
#results_ito = {}
#results_dicvol = {}
#results_treor = {}
results_gsas = {}
for index in tqdm(range(len(df))):
    entry = df.iloc[index]
    pattern = Path(entry.file_name).name.split('.')[0]

    #results_ito[pattern] = parse_ito(Path(ito_dir, pattern, f'{pattern}.imp'))

    #results_dicvol[pattern] = parse_dicvol(Path(dicvol_dir, pattern, f'{pattern}.imp'))

    #results_treor_high = parse_treor(Path(treor_dir, pattern, f'{pattern}_high.imp'))
    #results_treor_tric = parse_treor(Path(treor_dir, pattern, f'{pattern}_tric.imp'))
    #if not results_treor_high is None and not results_treor_tric is None:
    #    results_treor[pattern] = pd.concat((results_treor_high, results_treor_tric))
    #elif results_treor_high is None:
    #    results_treor[pattern] = results_treor_tric
    #elif results_treor_tric is None:
    #    results_treor[pattern] = results_treor_high

    results_gsas_high = parse_gsas(Path(gsas_dir, pattern, f'{pattern}.log'))
    results_gsas_tric = parse_gsas(Path(gsas_dir, pattern, f'{pattern}_triclinic.log'))
    if not results_gsas_high is None and not results_gsas_tric is None:
        results_gsas[pattern] = pd.concat((results_gsas_high, results_gsas_tric))
    elif results_gsas_high is None:
        results_gsas[pattern] = results_gsas_tric
    elif results_gsas_tric is None:
        results_gsas[pattern] = results_gsas_high

100%|██████████| 599/599 [00:22<00:00, 26.14it/s]


In [19]:
len(results_gsas)

599

In [20]:
results_gsas[list(results_gsas.keys())[0]]

,M20,spacegroup,a,b,c,alpha,beta,gamma,lattice_system,bravais_lattice
0,57.4518,C 1 2 1,10.08350,8.62580,13.37150,90.0,102.1190,90.0,monoclinic,mC
1,48.4059,C 2 2 2,4.43984,18.30680,12.99990,90.0,90.0000,90.0,orthorhombic,oC
2,48.1883,C 1 2 1,4.51389,14.95670,13.05050,90.0,90.4746,90.0,monoclinic,mC
3,41.8739,C 1 2 1,6.11787,6.10162,12.99700,90.0,90.3711,90.0,monoclinic,mC
4,41.6506,P 1 2 1,3.77540,13.01360,4.36040,90.0,98.2612,90.0,monoclinic,mP
...,...,...,...,...,...,...,...,...,...,...
84,10.8969,C 1 2 1,5.95435,6.32130,13.00440,90.0,90.9436,90.0,monoclinic,mC
85,10.8171,C 2 2 2,18.25680,18.68320,2.76202,90.0,90.0000,90.0,orthorhombic,oC
86,10.6099,C 2 2 2,8.62362,9.85248,13.02120,90.0,90.0000,90.0,orthorhombic,oC
87,10.3584,C 1 2 1,3.84176,26.03170,6.82068,90.0,101.1860,90.0,monoclinic,mC


In [9]:
results_dicvol[list(results_dicvol.keys())[0]]

,System,A,B,C,Alpha,Beta,Gamma,Volume,M
0,Orthorhombic,12.98126,12.94693,4.30791,90.00,90.000,90.000,724.02,13.8
1,Monoclinic,13.38430,4.31390,6.84010,NaN,103.637,NaN,383.81,13.6
2,Monoclinic,13.52380,4.31380,6.83870,NaN,105.850,NaN,383.79,15.7
3,Triclinic,13.45780,4.61500,4.46950,104.85,95.613,102.215,258.83,16.4


In [10]:
results_treor[list(results_treor.keys())[0]]

,lattice_system,A,B,C,Alpha,Beta,Gamma,Volume,M
0,tetragonal,18.310987,18.310987,3.901844,90.000000,90.000000,90.000000,1308.26,29
1,tetragonal,18.311302,18.311302,3.901826,90.000000,90.000000,90.000000,1308.30,25
0,triclinic,3.073150,4.347537,13.319566,91.563354,76.633255,96.777245,171.93,11


In [11]:
results_ito[list(results_ito.keys())[0]]

,Lines Indexed,Figure of Merit,Unit Cell,Volume,Centering
0,20,188.0,"[6.093, 12.933, 6.093, 90.0, 90.0, 90.0]",480.17,B
1,20,163.1,"[4.309, 12.933, 4.309, 90.0, 90.0, 90.0]",240.09,P
2,19,52.9,"[8.464, 13.111, 4.548, 93.055, 101.928, 98.245]",486.90,B
3,19,27.0,"[13.14, 10.556, 6.191, 90.0, 100.182, 90.0]",845.16,P
4,20,13.4,"[7.322, 13.19, 6.032, 97.876, 98.361, 80.874]",565.21,P
5,20,10.6,"[8.62, 51.895, 6.515, 90.0, 90.0, 90.0]",2914.48,P
6,20,10.4,"[8.62, 51.976, 4.454, 90.0, 90.0, 90.0]",1995.50,A
7,18,21.3,"[14.933, 25.931, 4.311, 90.0, 90.0, 90.0]",1669.17,B
8,20,4.5,"[8.286, 51.817, 6.677, 90.0, 90.0, 90.0]",2867.00,P


In [21]:
with pd.HDFStore("results/results_gsas.h5") as store:
    for key, df in results_gsas.items():
        if not df is None:
            store[key] = df